In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    p = _root / "paths.py"
    if p.is_file() and "CENTAUR_ROOT" in p.read_text(encoding="utf-8"):
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find CentaurLab_Analysis root (paths.py with CENTAUR_ROOT). "
        "Use Jupyter cwd CentaurLab_Analysis or CentaurLab_Analysis/notebooks."
    )
import paths

## Match 2k IDs with the 4k IDs

In [7]:
import pandas as pd


# Load the CSV file
df = pd.read_csv(paths.TABLES / "Centaur_Lab_Second_Round.csv")
df2k = pd.read_csv(paths.DATA / "merged_2k_with_4k_ID.csv")

# Display the first few rows after removing duplicates
# print(df_op4.columns)
print(df.head())
print(len(df))

   Origin       q2                  q3       q4       q5       q6  \
0  ID0002      1.0  0.8333333333333334  REMOVED      1.0  REMOVED   
1  ID0003      1.0                 1.0  REMOVED      1.0      1.0   
2  ID0007  REMOVED                 1.0      1.0      1.0      1.0   
3  ID0009  REMOVED                 1.0      1.0      1.0      1.0   
4  ID0010      1.0             REMOVED  REMOVED  REMOVED  REMOVED   

                   q7       q8       q9      q10  ... sentence_16 sentence_17  \
0  0.8333333333333334  REMOVED  REMOVED  REMOVED  ...         NaN         NaN   
1                 1.0  REMOVED      NaN      NaN  ...         NaN         NaN   
2             REMOVED      1.0      NaN      NaN  ...         NaN         NaN   
3                 1.0  REMOVED  REMOVED      NaN  ...         NaN         NaN   
4             REMOVED      1.0      NaN      NaN  ...         NaN         NaN   

  sentence_18 sentence_19 sentence_20 sentence_21 word_count number_count  \
0         NaN        

In [8]:
df_merged = (
    df
    .merge(
        df2k[["ID", "4k_ID"]],
        left_on="Origin",
        right_on="ID",
        how="left"
    )
    .drop(columns="ID")  # drop the redundant key column if desired
)

# Now df_merged contains all original df columns plus the new '4k_ID'
print(df_merged.head())
print(len(df_merged))

   Origin       q2                  q3       q4       q5       q6  \
0  ID0002      1.0  0.8333333333333334  REMOVED      1.0  REMOVED   
1  ID0003      1.0                 1.0  REMOVED      1.0      1.0   
2  ID0007  REMOVED                 1.0      1.0      1.0      1.0   
3  ID0009  REMOVED                 1.0      1.0      1.0      1.0   
4  ID0010      1.0             REMOVED  REMOVED  REMOVED  REMOVED   

                   q7       q8       q9      q10  ... sentence_17 sentence_18  \
0  0.8333333333333334  REMOVED  REMOVED  REMOVED  ...         NaN         NaN   
1                 1.0  REMOVED      NaN      NaN  ...         NaN         NaN   
2             REMOVED      1.0      NaN      NaN  ...         NaN         NaN   
3                 1.0  REMOVED  REMOVED      NaN  ...         NaN         NaN   
4             REMOVED      1.0      NaN      NaN  ...         NaN         NaN   

  sentence_19 sentence_20 sentence_21 word_count number_count  \
0         NaN         NaN        

In [9]:
import ast
import re

def extract_sentence_numbers(s: str) -> str:
    """
    Given a string like
    "['1. Foo', '2. Bar', '11. Baz']",
    return '1, 2, 11'.
    """
    try:
        # safely evaluate the list literal
        sentences = ast.literal_eval(s)
    except (ValueError, SyntaxError):
        return ""
    # extract the number before the first dot in each sentence
    nums = []
    for sent in sentences:
        m = re.match(r"\s*(\d+)\.", sent)
        if m:
            nums.append(m.group(1))
    # join into a comma‐separated string
    return ", ".join(nums)

# apply to your DataFrame
df_merged["human_sentence_ids"] = df_merged["New_Sentences"].apply(extract_sentence_numbers)

# Inspect
print(df_merged[["New_Sentences", "human_sentence_ids"]].head())

                                       New_Sentences human_sentence_ids
0  ['1. A woman in her 60s with a history of hype...     1, 2, 4, 6, 11
1  ['1. A 20-year-old woman comes to the primary ...      1, 2, 4, 5, 6
2  ['2. He has felt very weak every morning with ...      2, 3, 4, 5, 7
3  ['2. The lesions had been present since childh...      2, 3, 4, 5, 6
4  ['1. A 17-year-old high school student acciden...               1, 7


In [10]:
# Select the column as its own DataFrame and write to disk
df_merged[['4k_ID']].to_csv(paths.TABLES / '1300_IDs.csv', index=False)

In [11]:
import pandas as pd

# 1. Load your llama ContextCite CSV
llama_ContextCite = pd.read_csv(paths.TABLES / "llama70b_ContextCite_Merge.csv")

# 2. Merge in the 'Origin' column from df_merged where QA_ID == 4k_ID
llama_ContextCite = (
    llama_ContextCite
    .merge(
        df_merged[["4k_ID", "Origin"]],
        left_on="QA_ID",
        right_on="4k_ID",
        how="left"
    )
    .drop(columns="4k_ID")  # optional: remove the helper key
)

# 3. Inspect
print(llama_ContextCite.head())
print(f"Total rows: {len(llama_ContextCite)}")


                Score                                             Source  \
0    16.4903377201195  A man in his 30s with AIDS presented with acut...   
1    7.76135006235185  These contents were placed on a glass slide, f...   
2                 0.0  The patient also had a new-onset cough but was...   
3  -0.587592699838799                                                  .   
4   -2.12193597224481  For rapid bedside differentiation of multiple ...   

      QA_ID Extracted_Answer                         Raw_Response  Origin  
0  Merge Q1                D  <answer>Option D</answer><|im_end|>  ID1207  
1  Merge Q1                D  <answer>Option D</answer><|im_end|>  ID1207  
2  Merge Q1                D  <answer>Option D</answer><|im_end|>  ID1207  
3  Merge Q1                D  <answer>Option D</answer><|im_end|>  ID1207  
4  Merge Q1                D  <answer>Option D</answer><|im_end|>  ID1207  
Total rows: 31922


In [12]:
import pandas as pd

# assume df_merged and llama_ContextCite are already loaded

# 1. Extract the set of valid 4k_IDs (dropping any missing)
valid_ids = df_merged["4k_ID"]

# 2. Filter llama_ContextCite to keep only rows whose QA_ID is in that set
llama_ContextCite_filtered = llama_ContextCite[
    llama_ContextCite["QA_ID"].isin(valid_ids)
].reset_index(drop=True)

# 3. (Optional) inspect the result
print(f"Kept {len(llama_ContextCite_filtered)} rows out of {len(llama_ContextCite)}")
llama_ContextCite_filtered.head()

Kept 13016 rows out of 31922


,Score,Source,QA_ID,Extracted_Answer,Raw_Response,Origin
0,16.4903377201195,A man in his 30s with AIDS presented with acut...,Merge Q1,D,<answer>Option D</answer><|im_end|>,ID1207
1,7.76135006235185,"These contents were placed on a glass slide, f...",Merge Q1,D,<answer>Option D</answer><|im_end|>,ID1207
2,0.0,The patient also had a new-onset cough but was...,Merge Q1,D,<answer>Option D</answer><|im_end|>,ID1207
3,-0.587592699838799,.,Merge Q1,D,<answer>Option D</answer><|im_end|>,ID1207
4,-2.12193597224481,For rapid bedside differentiation of multiple ...,Merge Q1,D,<answer>Option D</answer><|im_end|>,ID1207


In [13]:
import pandas as pd

# 1. Ensure that 'Score' is numeric
llama_ContextCite_filtered['Score'] = pd.to_numeric(
    llama_ContextCite_filtered['Score'],
    errors='coerce'
)

# Optionally drop any rows where Score could not be parsed
llama_ContextCite_filtered = llama_ContextCite_filtered.dropna(subset=['Score'])

# 2. Recompute how many sentences to keep per QA_ID
df_merged['keep_k'] = (
    df_merged['sentence_number_corr']
    - df_merged['REMOVED_Sentences']
).clip(lower=0)  # prevent negative values
keep_k_map = df_merged.set_index('4k_ID')['keep_k'].to_dict()

# 3. Define a selector that picks the top-k rows by Score
def select_top_k(group: pd.DataFrame) -> pd.DataFrame:
    qa_id = group.name
    k = keep_k_map.get(qa_id, 0)
    if k <= 0:
        return group.iloc[0:0]  # empty DataFrame if nothing to keep
    return group.nlargest(k, 'Score')

# 4. Apply per-group filtering
llama_ContextCite_top = (
    llama_ContextCite_filtered
    .groupby('QA_ID', group_keys=False)
    .apply(select_top_k)
    .reset_index(drop=True)
)

# 5. Inspect how many rows remain
print(f"Rows before top-k filtering: {len(llama_ContextCite_filtered)}")
print(f"Rows after  top-k filtering: {len(llama_ContextCite_top)}")

Rows before top-k filtering: 13016
Rows after  top-k filtering: 6975


/tmp/ipykernel_4160450/2211860133.py:31: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(select_top_k)


In [14]:
llama_ContextCite_top

,Score,Source,QA_ID,Extracted_Answer,Raw_Response,Origin
0,16.490338,A man in his 30s with AIDS presented with acut...,Merge Q1,D,<answer>Option D</answer><|im_end|>,ID1207
1,7.761350,"These contents were placed on a glass slide, f...",Merge Q1,D,<answer>Option D</answer><|im_end|>,ID1207
2,0.000000,The patient also had a new-onset cough but was...,Merge Q1,D,<answer>Option D</answer><|im_end|>,ID1207
3,-0.587593,.,Merge Q1,D,<answer>Option D</answer><|im_end|>,ID1207
4,20.068556,Coagulation-related tests indicated a low clot...,Merge Q10,C,<answer>Option C</answer><|im_end|>,ID0589
...,...,...,...,...,...,...
6970,-1.048663,The rash was accompanied by diffuse alopecia o...,Merge Q995,B,<answer>Option B</answer><|im_end|>,ID1651
6971,-1.468974,A 50-year old man with a history of metastatic...,Merge Q995,B,<answer>Option B</answer><|im_end|>,ID1651
6972,7.035646,A plain radiograph of the left femur showed a ...,Merge Q996,A,<answer>Option A</answer><|im_end|>,ID0431
6973,2.087684,Serum protein electrophoresis testing did not ...,Merge Q996,A,<answer>Option A</answer><|im_end|>,ID0431


In [15]:
import pandas as pd

# Assume llama_ContextCite_top is already in scope

# 1. Group by QA_ID and join all Source strings
Llama_ContextCite_Removal = (
    llama_ContextCite_top
    .groupby("QA_ID", as_index=False)["Source"]
    .agg(context=lambda texts: " ".join(texts.astype(str)))
)

# 2. Inspect the new DataFrame
print(Llama_ContextCite_Removal.head(30))
print(len(Llama_ContextCite_Removal))

          QA_ID                                            context
0      Merge Q1  A man in his 30s with AIDS presented with acut...
1     Merge Q10  Coagulation-related tests indicated a low clot...
2    Merge Q100  Anterior segment optical coherence tomography ...
3   Merge Q1000  The findings of the remainder of the examinati...
4   Merge Q1001  B, Incisional biopsy reveals a necrotic epithe...
5   Merge Q1006  B, After 15 months, the lesions had evolved in...
6   Merge Q1007  There was no history of developmental delay, e...
7   Merge Q1008  Family history was notable for prostate cancer...
8    Merge Q101  Additionally, his daily activities were impair...
9   Merge Q1012  A- and B-scan ultrasonography demonstrated a c...
10  Merge Q1017  D, A Warthin-Starry silver impregnation stain ...
11  Merge Q1018  An ultrasound showed a right above-the-knee de...
12  Merge Q1019  aVF indicates augmented vector foot; aVL indic...
13   Merge Q102  A 3.5-mm left enophthalmos was noted compared

In [16]:
import pandas as pd

# 1. Restrict to QA_IDs that appear in df_merged’s 4k_ID
valid_ids = df_merged["4k_ID"].dropna().unique()
filtered = Llama_ContextCite_Removal[
    Llama_ContextCite_Removal["QA_ID"].isin(valid_ids)
]

# 2. Merge in the additional columns from df_merged
Llama_ContextCite_Removal = (
    filtered
    .merge(
        df_merged[["4k_ID", "question_options", "answer_df3", "data_source_corr", "Origin"]],
        left_on="QA_ID",
        right_on="4k_ID",
        how="left"
    )
    .drop(columns="4k_ID")
    .reset_index(drop=True)
)

# 3. (Optional) inspect the head
print(Llama_ContextCite_Removal.head())
print(len(Llama_ContextCite_Removal))

         QA_ID                                            context  \
0     Merge Q1  A man in his 30s with AIDS presented with acut...   
1    Merge Q10  Coagulation-related tests indicated a low clot...   
2   Merge Q100  Anterior segment optical coherence tomography ...   
3  Merge Q1000  The findings of the remainder of the examinati...   
4  Merge Q1001  B, Incisional biopsy reveals a necrotic epithe...   

                                    question_options answer_df3  \
0  What Is Your Diagnosis?\n\nA: Herpes simplex v...          D   
1  What Is Your Diagnosis?\n\nA: Pseudoxanthoma e...          C   
2  What Would You Do Next?\n\nA: Begin oral acycl...          D   
3  What Would You Do Next?\n\nA: Obtain a corneal...          B   
4  What Would You Do Next?\n\nA: Perform wide loc...          B   

  data_source_corr  Origin  
0             jama  ID1207  
1             jama  ID0589  
2             jama  ID1123  
3             jama  ID0021  
4             jama  ID0044  
1297


## Compare 70B sentences w/ Human sentences

In [17]:
import pandas as pd
import re

# 1. Merge in the step1_sentences column as before
sent_labels = pd.read_csv(paths.TABLES / "Sentence_Label_Original_2k.csv")
merged = Llama_ContextCite_Removal.merge(
    sent_labels[['ID', 'step1_sentences']],
    left_on='Origin',
    right_on='ID',
    how='left'
)

# 2. Define a function to extract and filter sentence numbers
def extract_70B_ids(row):
    context = row['context']
    step1   = row.get('step1_sentences', '')
    # Find all numbered sentences in the reference string
    pairs = re.findall(r'(\d+)\.\s*([^\.]+?)(?=\.|$)', str(step1))
    filtered = []
    for num_str, sentence in pairs:
        num = int(num_str)
        # only keep if 1 <= num <= 21 and sentence text appears in context
        if 1 <= num <= 21 and sentence.strip() in context:
            filtered.append(num)
    # Sort and format as "1. 2. 4. 5."
    filtered.sort()
    return ' '.join(f'{n}.' for n in filtered)

# 3. Apply and rebuild your final DataFrame
merged['70B_sentence_ids'] = merged.apply(extract_70B_ids, axis=1)
Llama_ContextCite_Removal = merged.drop(columns=['ID', 'step1_sentences']).reset_index(drop=True)
Llama_ContextCite_Removal

,QA_ID,context,question_options,answer_df3,data_source_corr,Origin,70B_sentence_ids
0,Merge Q1,A man in his 30s with AIDS presented with acut...,What Is Your Diagnosis?\n\nA: Herpes simplex v...,D,jama,ID1207,1. 2. 5.
1,Merge Q10,Coagulation-related tests indicated a low clot...,What Is Your Diagnosis?\n\nA: Pseudoxanthoma e...,C,jama,ID0589,2. 4. 6. 7. 8. 10. 12. 12. 12.
2,Merge Q100,Anterior segment optical coherence tomography ...,What Would You Do Next?\n\nA: Begin oral acycl...,D,jama,ID1123,3. 4. 5. 6. 7. 8. 9. 10. 11. 12.
3,Merge Q1000,The findings of the remainder of the examinati...,What Would You Do Next?\n\nA: Obtain a corneal...,B,jama,ID0021,1. 2. 3. 4. 6. 7. 8. 9. 10. 11.
4,Merge Q1001,"B, Incisional biopsy reveals a necrotic epithe...",What Would You Do Next?\n\nA: Perform wide loc...,B,jama,ID0044,2. 3. 4. 5. 8. 11. 12. 13.
...,...,...,...,...,...,...,...
1292,Merge Q990,"C, Fluorescein angiography demonstrates hyperf...",What Would You Do Next?\n\nA: Consult otolaryn...,C,jama,ID0467,1. 6. 11. 14. 15.
1293,Merge Q992,Stains were weak for CD4 and were negative for...,What Is Your Diagnosis?\n\nA: Disseminated int...,C,jama,ID0462,4. 5.
1294,Merge Q994,"Multiple temperature measurements, including a...",What Would You Do Next?\n\nA: Order an emergen...,C,jama,ID0487,2. 3.
1295,Merge Q995,He underwent colostomy after unsuccessful surg...,What Would You Do Next?\n\nA: Perform skin bio...,B,jama,ID1651,1. 2. 5. 6. 7. 8. 9.


In [18]:
# Save the filtered context‐citation DataFrame to disk
Llama_ContextCite_Removal.to_csv(paths.TABLES / "70B_ContextCite_Removal.csv", index=False)
print("Saved 70B_ContextCite_Removal.csv")


Saved 70B_ContextCite_Removal.csv


In [19]:
import pandas as pd
import re

# Reload your sentence‐label reference
sent_labels = pd.read_csv(paths.TABLES / "Sentence_Label_Original_2k.csv")[['ID', 'step1_sentences']]

# 1. Re‐merge to recover 'step1_sentences'
temp = (
    Llama_ContextCite_Removal
    .merge(
        sent_labels,
        left_on='Origin',
        right_on='ID',
        how='left'
    )
)

# 2. Helper to extract the set of matched IDs from the string "1. 2. 4. 5."
def parse_matched_ids(id_str: str) -> set[int]:
    return set(int(n) for n in re.findall(r'(\d+)', str(id_str)))

# 3. Build the "70B_After_Removal" entries
def build_after_removal(row) -> str:
    # Parse the original (number, sentence) pairs
    pairs = re.findall(r'(\d+)\.\s*([^\.]+?)(?=\.|$)', str(row['step1_sentences']))
    matched = parse_matched_ids(row['70B_sentence_ids'])
    # Keep only those in matched, preserving original order
    kept = [
        f"{num}. {sent.strip()}"
        for num, sent in pairs
        if int(num) in matched
    ]
    # Join with a space (or '\n' if you prefer line breaks)
    return " ".join(kept)

# 4. Apply and assign the new column
temp['70B_After_Removal'] = temp.apply(build_after_removal, axis=1)

# 5. Drop helper columns if desired
Llama_ContextCite_Removal = temp.drop(columns=['ID', 'step1_sentences']).reset_index(drop=True)

# Inspect the result
print(Llama_ContextCite_Removal[['Origin', '70B_sentence_ids', '70B_After_Removal']].head())

   Origin                  70B_sentence_ids  \
0  ID1207                          1. 2. 5.   
1  ID0589    2. 4. 6. 7. 8. 10. 12. 12. 12.   
2  ID1123  3. 4. 5. 6. 7. 8. 9. 10. 11. 12.   
3  ID0021   1. 2. 3. 4. 6. 7. 8. 9. 10. 11.   
4  ID0044        2. 3. 4. 5. 8. 11. 12. 13.   

                                   70B_After_Removal  
0  1. A man in his 30s with AIDS presented with a...  
1  2. In early childhood, she had developed asymp...  
2  3. Ocular history was notable for neovascular ...  
3  1. An adolescent boy was referred to the emerg...  
4  2. This was accompanied by foreign-body sensat...  


In [23]:
import pandas as pd
import re
import numpy as np

# 1. First, merge your two DataFrames on 'Origin' to get both columns side by side
comparison_df = pd.merge(
    df_merged[['Origin', 'human_sentence_ids', 'data_source_corr']],
    Llama_ContextCite_Removal[['Origin', '70B_sentence_ids']],
    on='Origin',
    how='inner'
)

# 2. Helper to parse a string like "1,2,5" or "1. 3. 5" into a set of integers
def parse_ids(id_str: str) -> set:
    if pd.isna(id_str) or not id_str:
        return set()
    # find all consecutive digits
    return set(int(n) for n in re.findall(r'(\d+)', id_str))

# 3. Compute match rate = |intersection| / |model_ids|
def compute_match_rate(row) -> float:
    human_ids = parse_ids(row['human_sentence_ids'])
    model_ids = parse_ids(row['70B_sentence_ids'])
    if not model_ids:
        return np.nan  # or 0.0, depending on how you want to handle empty
    return len(human_ids & model_ids) / len(model_ids)

# 4. Apply to each row and store in a new column
comparison_df['match_rate_70B'] = comparison_df.apply(compute_match_rate, axis=1)

# 5. (Optional) View the results
print(comparison_df[['Origin', 'human_sentence_ids', '70B_sentence_ids', 'match_rate_70B', 'data_source_corr']])


# 1. Full descriptive statistics
stats = comparison_df['match_rate_70B'].describe()
print("Descriptive statistics for match_rate_70B:")
print(stats)

# 2. Mean and standard deviation as percentages
mean_rate = comparison_df['match_rate_70B'].mean()
std_rate  = comparison_df['match_rate_70B'].std()
print(f"\nMean match rate      : {mean_rate:.2%}")
print(f"Std. dev. of match rate: {std_rate:.2%}")

      Origin human_sentence_ids   70B_sentence_ids  match_rate_70B  \
0     ID0002     1, 2, 4, 6, 11     4. 6. 7. 8. 9.        0.400000   
1     ID0003      1, 2, 4, 5, 6     1. 2. 5. 6. 7.        0.800000   
2     ID0007      2, 3, 4, 5, 7     1. 4. 5. 6. 7.        0.600000   
3     ID0009      2, 3, 4, 5, 6  1. 3. 5. 6. 7. 8.        0.500000   
4     ID0010               1, 7              4. 7.        0.500000   
...      ...                ...                ...             ...   
1292  ID1995            1, 2, 5           1. 4. 5.        0.666667   
1293  ID1996               1, 3                                NaN   
1294  ID1997            1, 2, 4          1. 3. 10.        0.333333   
1295  ID1998   1, 2, 3, 4, 5, 6  1. 2. 5. 6. 7. 8.        0.666667   
1296  ID1999            1, 2, 5              3. 4.        0.000000   

     data_source_corr  
0                jama  
1            medxpert  
2          medbullets  
3                jama  
4            medxpert  
...            

In [26]:
# 1. Group by 'data_source_corr' and calculate mean (accuracy) and standard deviation (std)
data_source_stats = comparison_df.groupby('data_source_corr')['match_rate_70B'].agg(['mean', 'std']).reset_index()

# 2. Rename columns for clarity
data_source_stats.columns = ['data_source_corr', 'Accuracy (Mean)', 'Standard Deviation (Std)']

# 3. Display the results
data_source_stats


,data_source_corr,Accuracy (Mean),Standard Deviation (Std)
0,jama,0.621317,0.225943
1,medbullets,0.666347,0.235701
2,medxpert,0.693016,0.271610
3,mmlu,0.707212,0.247309


In [21]:
# pip install krippendorff
import re
import pandas as pd
import krippendorff

# parse strings like "1, 2, 4. 6. 7." into [1,2,4,6,7]
def parse_ids(cell):
    return [int(n) for n in re.findall(r'\d+', cell)]

# assume comparison_df has the two original columns
comparison_df['human_ids']   = comparison_df['human_sentence_ids'].apply(parse_ids)
comparison_df['model70_ids'] = comparison_df['70B_sentence_ids'].apply(parse_ids)

def kripp_alpha_row(row):
    # build the union of sentence IDs
    all_ids = sorted(set(row['human_ids'] + row['model70_ids']))
    # for each coder, mark 1 if they selected that ID, else 0
    human_binary = [1 if sid in row['human_ids']   else 0 for sid in all_ids]
    model_binary = [1 if sid in row['model70_ids'] else 0 for sid in all_ids]
    # stack as a list of lists: [coder1_annotations, coder2_annotations]
    data_matrix = [human_binary, model_binary]
    # compute nominal‐level Krippendorff’s alpha
    return krippendorff.alpha(reliability_data=data_matrix,
                              level_of_measurement='nominal')

comparison_df['kripp_alpha'] = comparison_df.apply(kripp_alpha_row, axis=1)

print(comparison_df[['human_ids', 'model70_ids', 'kripp_alpha']])


ModuleNotFoundError: No module named 'krippendorff'

In [86]:
import pandas as pd

# Assuming df_merged and Llama_ContextCite_Removal are already in scope

# 1. Map human_sentence_ids into Llama_ContextCite_Removal via the Origin key
human_map = df_merged.set_index("Origin")["human_sentence_ids"]
Llama_ContextCite_Removal["human_sentence_ids"] = (
    Llama_ContextCite_Removal["Origin"]
    .map(human_map)
)

# 2. Save to CSV
Llama_ContextCite_Removal.to_csv(paths.TABLES / "70B_ContextCite_Removal.csv", index=False)
print("Saved 70B_ContextCite_Removal.csv with human_sentence_ids column.")

Saved 70B_ContextCite_Removal.csv with human_sentence_ids column.
